# DS 4021 Final Project: SUPPORT 2 Study on End-of-Life Care

Loading in Data:

In [1]:
!pip install ucimlrepo

In [2]:
import pandas as pd
from ucimlrepo import fetch_ucirepo

support2 = fetch_ucirepo(id=880)

df = support2.data.original.copy()


print(df.shape)
df.head()

(9105, 48)


,id,age,death,sex,hospdead,slos,d.time,dzgroup,dzclass,num.co,...,crea,sod,ph,glucose,bun,urine,adlp,adls,sfdm2,adlsc
0,1,62.84998,0,male,0,5,2029,Lung Cancer,Cancer,0,...,1.199951,141.0,7.459961,NaN,NaN,NaN,7.0,7.0,NaN,7.0
1,2,60.33899,1,female,1,4,4,Cirrhosis,COPD/CHF/Cirrhosis,2,...,5.500000,132.0,7.250000,NaN,NaN,NaN,NaN,1.0,<2 mo. follow-up,1.0
2,3,52.74698,1,female,0,17,47,Cirrhosis,COPD/CHF/Cirrhosis,2,...,2.000000,134.0,7.459961,NaN,NaN,NaN,1.0,0.0,<2 mo. follow-up,0.0
3,4,42.38498,1,female,0,3,133,Lung Cancer,Cancer,2,...,0.799927,139.0,NaN,NaN,NaN,NaN,0.0,0.0,no(M2 and SIP pres),0.0
4,5,79.88495,0,female,0,16,2029,ARF/MOSF w/Sepsis,ARF/MOSF,1,...,0.799927,143.0,7.509766,NaN,NaN,NaN,NaN,2.0,no(M2 and SIP pres),2.0


In [ ]:
# See how UCI labeled each column (feature, target, or ID) and its description
support2.variables

,name,role,type,demographic,description,units,missing_values
0,id,ID,Integer,None,None,None,no
1,age,Feature,Continuous,Age,Age of the patients in years,years,no
2,death,Target,Continuous,None,Death at any time up to National Death Index (...,None,no
3,sex,Feature,Categorical,Sex,Gender of the patient. Listed values are {male...,None,no
4,hospdead,Target,Binary,None,Death in hospital,None,no
5,slos,Other,Continuous,None,Days from Study Entry to Discharge,None,no
6,d.time,Other,Continuous,None,Days of follow-up,None,no
7,dzgroup,Feature,Categorical,None,The patient's disease sub category amogst ARF/...,None,no
8,dzclass,Feature,Categorical,None,"The patient's disease category amongst ""ARF/MO...",None,no
9,num.co,Feature,Continuous,None,The number of simultaneous diseases (or comorb...,None,no


In [ ]:
df.info()                              # data types and non-missing counts
df.isna().mean().sort_values(ascending=False).head(15)  # columns with the most missing data

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9105 entries, 0 to 9104
Data columns (total 45 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       9105 non-null   float64
 1   sex       9105 non-null   object 
 2   dzgroup   9105 non-null   object 
 3   dzclass   9105 non-null   object 
 4   num.co    9105 non-null   int64  
 5   edu       7471 non-null   float64
 6   income    6123 non-null   object 
 7   scoma     9104 non-null   float64
 8   charges   8933 non-null   float64
 9   totcst    8217 non-null   float64
 10  totmcst   5630 non-null   float64
 11  avtisst   9023 non-null   float64
 12  race      9063 non-null   object 
 13  sps       9104 non-null   float64
 14  aps       9104 non-null   float64
 15  surv2m    9104 non-null   float64
 16  surv6m    9104 non-null   float64
 17  hday      9105 non-null   int64  
 18  diabetes  9105 non-null   int64  
 19  dementia  9105 non-null   int64  
 20  ca        9105 non-null   obje

,0
adlp,0.619550
urine,0.533992
glucose,0.494234
bun,0.477979
totmcst,0.381658
alb,0.370346
income,0.327512
adls,0.314882
bili,0.285667
pafi,0.255354


## Creating Target Variable

In [4]:
# 1. Count censored patients: alive at last contact but followed for fewer than 180 days
censored = (df["death"] == 0) & (df["d.time"] < 180)
print("Censored patients:", censored.sum())

# 2. Drop them, since we can't tell whether they survived to 6 months
df = df[~censored].reset_index(drop=True)

# 3. Create the label: 1 = died within 180 days, 0 = alive at day 180
df["died_6m"] = ((df["death"] == 1) & (df["d.time"] <= 180)).astype(int)

print("Patients remaining:", len(df))
print(df["died_6m"].value_counts())
print(df["died_6m"].value_counts(normalize=True).round(3))

Censored patients: 0
Patients remaining: 9105
died_6m
0    4840
1    4265
Name: count, dtype: int64
died_6m
0    0.532
1    0.468
Name: proportion, dtype: float64


Dataset Description:

In [5]:
print("Sample size:", len(df))
print("Variables (excluding ID):", df.shape[1] - 2)   # minus id and the died_6m column we added

# Candidate predictors: everything except outcomes, post-day-3 info, and prior model/physician estimates
drop = ["id", "died_6m", "death", "hospdead", "sfdm2", "slos", "d.time",
        "charges", "totcst", "totmcst", "avtisst",
        "sps", "aps", "surv2m", "surv6m", "prg2m", "prg6m", "dnr", "dnrday"]
predictors = [c for c in df.columns if c not in drop]
print("Candidate predictors:", len(predictors))

print("\nAge: mean = %.1f, SD = %.1f" % (df["age"].mean(), df["age"].std()))
print("Education (years): mean = %.1f, SD = %.1f, missing = %d"
      % (df["edu"].mean(), df["edu"].std(), df["edu"].isna().sum()))
print("\nSex:\n", (df["sex"].value_counts(normalize=True) * 100).round(1))
print("\nRace:\n", (df["race"].value_counts(normalize=True) * 100).round(1))
print("\nDisease class:\n", (df["dzclass"].value_counts(normalize=True) * 100).round(1))
print("\nDied within 6 months: %.1f%%" % (df["died_6m"].mean() * 100))

Sample size: 9105
Variables (excluding ID): 47
Candidate predictors: 30

Age: mean = 62.7, SD = 15.6
Education (years): mean = 11.7, SD = 3.4, missing = 1634

Sex:
 sex
male      56.3
female    43.7
Name: proportion, dtype: float64

Race:
 race
white       79.3
black       15.3
hispanic     3.2
other        1.2
asian        0.9
Name: proportion, dtype: float64

Disease class:
 dzclass
ARF/MOSF              46.4
COPD/CHF/Cirrhosis    31.4
Cancer                15.6
Coma                   6.5
Name: proportion, dtype: float64

Died within 6 months: 46.8%


## Data Split

Creating the training and reporting set (80/20 split)

In [6]:
from sklearn.model_selection import train_test_split

train_df, report_df = train_test_split(
    df,
    test_size=0.20,            # 20% reporting set
    stratify=df["died_6m"],    # same death rate in both sets
    random_state=42            # fixed seed so the split can be recreated exactly
)

print("Train:", train_df.shape, " Reporting:", report_df.shape)
print("6-month death rate, train:     %.3f" % train_df["died_6m"].mean())
print("6-month death rate, reporting: %.3f" % report_df["died_6m"].mean())

train_df.to_csv("support2_train.csv", index=False)
report_df.to_csv("support2_reporting.csv", index=False)

Train: (7284, 49)  Reporting: (1821, 49)
6-month death rate, train:     0.468
6-month death rate, reporting: 0.468
